# Markowitz vs 1/N: walkthrough
Frontier, estimation error, then a walk-forward back-test of seven allocation rules. All logic lives in the `portopt` package.

In [1]:
SOURCE = "yahoo"
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from portopt import data as D, optimize as opt
pd.set_option('display.precision', 3)
R = (D.to_returns(D.load_prices(cache_dir='../data/raw')) if SOURCE == 'yahoo'
     else D.make_synthetic(4000))
mu, cov = opt.sample_moments(R)
pd.DataFrame({'mu_%': 100 * mu, 'vol_%': 100 * np.sqrt(np.diag(cov))}, index=R.columns)

,mu_%,vol_%
Ticker,,
SPY,11.985,19.056
QQQ,16.240,21.641
IWM,10.739,24.157
EFA,7.968,21.082
EEM,10.130,27.613
VNQ,10.660,28.758
TLT,4.263,14.677
IEF,3.492,6.791
LQD,4.327,8.537


## One covariance, five allocations

In [2]:
lw = opt.ledoit_wolf_cov(R)
pd.DataFrame({
    '1/N': opt.equal_weight(len(mu)),
    'MinVar': opt.min_variance(lw, (0, 0.4)),
    'MaxSharpe': opt.max_sharpe(mu, cov, bounds=(0, 0.4)),
    'ERC': opt.risk_parity(lw),
    'HRP': opt.hrp(cov),
}, index=R.columns).mul(100).round(1)

,1/N,MinVar,MaxSharpe,ERC,HRP
Ticker,,,,,
SPY,10.0,13.6,0.0,6.2,2.0
QQQ,10.0,0.0,27.3,5.7,1.5
IWM,10.0,0.0,0.0,5.1,2.3
EFA,10.0,0.0,0.0,5.5,2.9
EEM,10.0,0.0,0.0,4.4,1.7
VNQ,10.0,0.0,0.0,4.4,2.3
TLT,10.0,0.0,2.3,14.5,9.0
IEF,10.0,40.0,40.0,29.1,36.6
LQD,10.0,37.9,7.3,14.6,33.8


Risk parity check: every asset contributes 10% of the variance.

In [3]:
(100 * opt.risk_contributions(opt.risk_parity(lw), lw)).round(2)

array([10., 10., 10., 10., 10., 10., 10., 10., 10., 10.])

## Estimation error: promised vs realised

In [4]:
from portopt.frontier import split_frontier
sf = split_frontier(R)
{k: {s: round(v[0] / v[1], 2) for s, v in sf[k].items()} for k in ('tangency', 'equal_weight')}

{'tangency': {'promised': np.float64(1.3), 'realised': np.float64(0.97)},
 'equal_weight': {'promised': np.float64(0.66), 'realised': np.float64(0.77)}}

## Walk-forward back-test (monthly, 3-year window, 10 bp costs)

In [5]:
from portopt.backtest import default_strategies, target_weights, simulate
from portopt.metrics import performance_table, tests_vs_benchmark
sim = simulate(R, target_weights(R, default_strategies()), cost_bps=10)
performance_table(sim)

,return_%,vol_%,sharpe,sortino,max_dd_%,calmar,turnover_pa_%,var99_%,es99_%
strategy,,,,,,,,,
Equal weight (1/N),7.676,13.139,0.629,0.892,-36.699,0.209,31.638,2.351,3.460
Min variance (sample),5.546,6.695,0.840,1.203,-21.498,0.258,45.139,1.127,1.622
Min variance (Ledoit-Wolf),5.553,6.707,0.839,1.203,-21.636,0.257,44.203,1.132,1.622
Max Sharpe (sample),7.505,9.188,0.834,1.199,-26.189,0.287,237.284,1.632,2.112
Max Sharpe (Bayes-Stein + LW),7.572,8.453,0.906,1.317,-23.108,0.328,169.647,1.493,2.063
Risk parity (ERC),6.334,8.017,0.806,1.153,-24.044,0.263,30.108,1.340,1.895
Hierarchical risk parity,4.992,7.041,0.727,1.038,-22.883,0.218,41.967,1.214,1.646


In [6]:
tests_vs_benchmark(sim, 'Equal weight (1/N)', n_boot=1000)

,sharpe_diff,ci_low,ci_high,p_value
strategy,,,,
Min variance (sample),0.211,-0.103,0.568,0.237
Min variance (Ledoit-Wolf),0.211,-0.101,0.568,0.234
Max Sharpe (sample),0.205,-0.139,0.580,0.279
Max Sharpe (Bayes-Stein + LW),0.277,-0.045,0.625,0.101
Risk parity (ERC),0.178,-0.079,0.432,0.153
Hierarchical risk parity,0.098,-0.299,0.503,0.642
